# IVP-MCQ zero-shot baseline (letter log-probs)

Runs `baseline_letter_logprobs.py` on a Colab T4. Upload the `Thesis` folder (at least
`baseline_letter_logprobs.py` and `ivp_mcq/`) to Google Drive and set `PROJECT_DIR`.
Results go to `results/baseline/<timestamp>_<model>_<split>/` on Drive; nothing is overwritten.

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/Thesis'   # <- edit
%cd {PROJECT_DIR}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Qwen3 / Phi-4-mini / Gemma 3 need transformers >= 4.51 (Colab's is usually newer)
!pip -q install "transformers>=4.51" accelerate

## Hugging Face token (only for gated models: Llama 3.2, Gemma)
1. On huggingface.co, open each model page and accept the licence
   (meta-llama/Llama-3.2-1B-Instruct, meta-llama/Llama-3.2-3B-Instruct,
   google/gemma-2-2b-it, google/gemma-3-1b-it, google/gemma-3-4b-it).
2. Settings → Access Tokens → create a **Read** token.
3. In Colab, click the key icon (Secrets) in the left bar → add `HF_TOKEN` with the
   token as value → switch on *Notebook access*.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set')
except Exception as e:
    print('no HF_TOKEN secret (fine for non-gated models):', type(e).__name__)

## Settings
Thesis models and their dev-chosen prompts (see CLAUDE.md): Qwen2.5-3B-Instruct with
prompt v1, Phi-4-mini with the `letter-only` system message. `SPLIT='test'` is the reported
run; use `'dev'` only for prompt checks. To compare prompts on dev, list several, e.g.
`[None, 'letter-only']`.

On *CUDA out of memory*, lower `BATCH_SIZE`.

In [ ]:
SPLIT = 'test'
RUNS = {                       # model key -> system prompts to run (None = prompt v1)
    'qwen2.5-3b-instruct': [None],
    'phi-4-mini-instruct': ['letter-only'],
    # 'llama-3.2-3b-instruct': [None, 'letter-only'],   # once HF access is granted
}
BATCH_SIZE = 8
EXTRA = ''

In [ ]:
import shlex, subprocess
failed = []
for m, systems in RUNS.items():
    for sysm in systems:
        cmd = (f'python baseline_letter_logprobs.py --model {m} --split {SPLIT} '
               f'--batch-size {BATCH_SIZE} {EXTRA}' + (f' --system {shlex.quote(sysm)}' if sysm else ''))
        print('>>', cmd, flush=True)
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        out = [l for l in r.stdout.splitlines() if 'acc=' in l]
        print(out[-1] if out else '', flush=True)
        if r.returncode:
            failed.append((m, sysm))
            print('FAILED:' + chr(10) + chr(10).join(r.stderr.strip().splitlines()[-6:]), flush=True)
print('failed runs:', failed or 'none')

In [ ]:
# the two thesis models only (earlier runs of dropped models are also in results/)
!python baseline_letter_logprobs.py --table "results/baseline/*qwen2.5-3b-instruct*_{SPLIT}/summary.json" "results/baseline/*phi-4-mini-instruct*_{SPLIT}/summary.json"